In [ ]:
!pip install langchain pypdf --quiet
!pip install langchain-community --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.0/298.0 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 4.1 MB/s eta 0:00:00


In [ ]:
pip install pypdf2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 9.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from transformers import pipeline
import torch
from langchain.document_loaders import PyPDFLoader
from tqdm import tqdm
import re

In [ ]:
from PyPDF2 import PdfMerger

# List of PDF files to merge (provide the paths to your 4 PDFs)
pdf_files = [
    "/content/SecuritiesandCommoditiesExchangeMarketRelatedLaws.pdf",
    "/content/SpecializedInvestmentFundRules.pdf",
    "/content/StrategicPlan.pdf",
    "/content/booklet.pdf",
]

# Initialize the PdfMerger object
merger = PdfMerger()

# Loop through the PDF files and append them
for pdf in pdf_files:
    merger.append(pdf)

# Specify the output file name
output_file = "merged_document.pdf"

# Write the merged PDF to the output file
merger.write(output_file)
merger.close()

print(f"All PDFs have been merged into {output_file}")


All PDFs have been merged into merged_document.pdf


In [ ]:

# Load the merged PDF
pdf_name = "merged_document.pdf"
loader = PyPDFLoader(pdf_name)

# Split the PDF into pages
pages = loader.load_and_split()

print(f"Loaded {len(pages)} pages from the merged PDF.")

Loaded 998 pages from the merged PDF.


In [ ]:
def clean_dataset(entire_text):
    """
    Cleans the text content of a document page by performing multiple preprocessing steps:
    - Removes extra whitespaces.
    - Fixes hyphenated words.
    - Removes headers, footers, page numbers, and non-informative text.
    - Extracts relevant sections.
    - Normalizes and standardizes text.
    - Cleans unnecessary legal jargon.
    """
    text = entire_text.page_content

    # Step 1: Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Step 2: Fix hyphenated words (joins split words)
    text = re.compile(r'(\w+)\s*-\s*(\w+)').sub(lambda match: match.group(1) + match.group(2), text)

    # Step 3: Remove non-informative headers/footers like page numbers and ToC references
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)  # Page numbers
    text = re.sub(r"Table of Content.*", "", text, flags=re.IGNORECASE)  # ToC references

    # Step 4: Normalize and standardize text
    text = text.lower()  # Convert to lowercase for consistency
    text = text.replace("’", "'").replace("“", '"').replace("”", '"')  # Replace unusual quotation marks

    # Step 5: Remove redundant legal formatting
    text = re.sub(r"provided that.*?\.", "", text, flags=re.IGNORECASE)  # Remove "Provided that..." phrases

    # Step 6: Extract relevant sections (optional, if specific keywords are required)
    relevant_sections = ["securities act", "market regulation", "money laundering"]
    if not any(section in text for section in relevant_sections):
        return None  # Skip this page if no relevant section matches

    # Final Step: Update the cleaned text back to the page content
    entire_text.page_content = text.strip()
    return entire_text


In [ ]:
# Clean each page
cleaned_pages = [page for page in (clean_dataset(page) for page in pages) if page is not None]

# Check the first cleaned page for verification
print(cleaned_pages[0].page_content)


7 1. acts a. securities and commodities market i. securities act, 2006 (2063) ii. commodities exchange market act, 2017 (2074) b. money laundering i. asset (money) laundering prevention act, 2008 (2064)


In [ ]:
from langchain.text_splitter import CharacterTextSplitter

# Join all cleaned pages into one string
cleaned_text = "\n".join(page.page_content for page in cleaned_pages)

# Set up the text splitter
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separator="."
)

# Split the joined cleaned text into chunks
chunks = text_splitter.split_text(cleaned_text)

# Check the first chunk for verification
print(chunks[0])


7 1. acts a. securities and commodities market i. securities act, 2006 (2063) ii. commodities exchange market act, 2017 (2074) b. money laundering i


In [ ]:
len(chunks)

237

In [ ]:
!pip install sentence-transformers --quiet
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
# Initialize an instance of HuggingFaceEmbeddings with the specified parameters
sentence_transformer = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-mpnet-base-v2',# 'sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device':'cuda'},
)

<ipython-input-14-690859390d65>:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  sentence_transformer = HuggingFaceEmbeddings(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 73.4 MB/s eta 0:00:00


In [ ]:
from langchain.vectorstores import FAISS

In [ ]:
from langchain.schema import Document
# Convert chunks (strings) into Document objects
documents = [Document(page_content=chunk) for chunk in chunks]

In [ ]:
%%time

# Creating and storing embeddings in the FAISS vector store
vector_db = FAISS.from_documents(documents, sentence_transformer)

CPU times: user 4.94 s, sys: 151 ms, total: 5.09 s
Wall time: 5.59 s


In [ ]:
vector_db.save_local("final_ready_vector_db_data")

In [ ]:

# Load the FAISS vector store
docsearch = FAISS.load_local(
    "final_ready_vector_db_data",
    sentence_transformer,
    allow_dangerous_deserialization=True
)

# Perform a similarity search
query = "What is NEPSE?"
results = docsearch.similarity_search(query, k=5)

# Display the results
for result in results:
    print(result.page_content)

c) rules mean the rules framed under the act. d) nepse means the nepal stock exchange ltd. licensed by the board to run securities market. e) body corporate means a body corporate enliste d for securities at the nepse. f) institutional activities mean activities such as closing the registry book of a body corporate, organize general meetings, make retur n payments for the time expired s ecurities, revise the debentures and title deeds having convertible features, disbursing dividends, interest, bonus shares, right shares and priority shares, issue and pay f or the title deeds, return payment of premium and other associated tasks. g) institutional benefits mean benefits such as disbursing dividends, interest, bonus shares, rights shares, priority shares, issuing title deeds and making return payment of premium and other associated perks. h) enlistment means an enlistment made at the nepse for the purpose of purchase, sale or exchange of securities through the securities market
c) rules 